In [1]:
import torch
from torch import nn
from torch.func import vmap, grad, jacrev, jacfwd
import torch.utils.benchmark as benchmark
from networks.basics import MLP

In [2]:
# Ensure that all operations are deterministic on GPU (if used) for reproducibility
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Fetching the device that will be used throughout this notebook
device = torch.device("cpu") if not torch.cuda.is_available() else torch.device("cuda:0")
print("Using device", device)

Using device cuda:0


## Compare 'matmul' and elementwise multiplication

In [3]:
dadz = torch.randn(100, 5, device=device)   # (bs, n_hidden)
dyda = torch.randn(3, 5, device=device)     # (n_out, n_hidden)

out1 = torch.matmul(dyda, torch.diag_embed(dadz))  # (bs, n_out, n_hidden)
out2 = dadz[:, None, :] * dyda                     # (bs, n_out, n_hidden)
assert torch.allclose(out1, out2)
print(out1.shape)

t1 = benchmark.Timer(
    stmt="torch.matmul(dyda, torch.diag_embed(dadz))",
    globals={"dyda": dyda, "dadz": dadz})

t2 = benchmark.Timer(
    stmt="dadz[:, None, :] * dyda",
    globals={"dyda": dyda, "dadz": dadz})

print(t1.timeit(1000))
print(t2.timeit(1000))

torch.Size([100, 3, 5])
torch.matmul(dyda, torch.diag_embed(dadz))
  44.29 us
  1 measurement, 1000 runs , 1 thread
dadz[:, None, :] * dyda
  22.67 us
  1 measurement, 1000 runs , 1 thread


## Compare matmul and einsum

In [4]:
dydz = torch.randn(100, 5, 5, device=device)    # (bs, n_out, n_hidden)
dzdx = torch.randn(5, 5, device=device)         # (n_hidden, n_in)

out1 = torch.matmul(dydz, dzdx)                 # (bs, n_out, n_in)
out2 = torch.einsum("bik,kj->bij", dydz, dzdx)  # (bs, n_out, n_in)
assert torch.allclose(out1, out2)
print(out1.shape)

t1 = benchmark.Timer(
    stmt="torch.matmul(dydz, dzdx)",
    globals={"dydz": dydz, "dzdx": dzdx})

t2 = benchmark.Timer(
    stmt="torch.einsum('bik,kj->bij', dydz, dzdx)",
    globals={"dydz": dydz, "dzdx": dzdx})

print(t1.timeit(1000))
print(t2.timeit(1000))

torch.Size([100, 5, 5])
torch.matmul(dydz, dzdx)
  17.06 us
  1 measurement, 1000 runs , 1 thread
torch.einsum('bik,kj->bij', dydz, dzdx)
  40.85 us
  1 measurement, 1000 runs , 1 thread


## Compute Jacobian with autodiff

In [11]:
def func1(x, mlp):
    return vmap(grad(lambda p: mlp(p).squeeze()))(x)  # (bs, n_in)

def func2(x, mlp):
    return vmap(jacrev(mlp))(x)  # (bs, n_out, n_in)

def func3(x, mlp):
    jac = jacrev(mlp)(x)  # (bs, n_out, bs, n_in)
    jac = torch.diagonal(jac, dim1=0, dim2=2)  # (n_out, n_in, bs)
    jac = torch.permute(jac, (2, 0, 1))  # (bs, n_out, n_in)
    return jac

def func4(x, mlp):
    return vmap(jacfwd(mlp))(x)  # (bs, n_out, n_in)

def func5(x, mlp):
    jac = jacfwd(mlp)(x)  # (bs, n_out, bs, n_in)
    jac = torch.diagonal(jac, dim1=0, dim2=2)  # (n_out, n_in, bs)
    jac = torch.permute(jac, (2, 0, 1))  # (bs, n_out, n_in)
    return jac

In [12]:
x = torch.randn(100, 2, device=device, requires_grad=False)

mlp = MLP(layer_sizes=[2, 10, 1], activation=nn.Sigmoid())
mlp.to(device)
mlp

MLP(
  (layers): ModuleList(
    (0): Linear(in_features=2, out_features=10, bias=True)
    (1): Linear(in_features=10, out_features=1, bias=True)
  )
  (activation): Sigmoid()
)

In [13]:
jac1 = func1(x, mlp)
jac2 = func2(x, mlp)
jac3 = func3(x, mlp)
jac4 = func4(x, mlp)
jac5 = func5(x, mlp)

print(jac1.shape)
print(jac2.shape)
print(jac3.shape)
print(jac4.shape)
print(jac5.shape)

print(torch.linalg.norm(jac2.squeeze(-2) - jac1))
assert torch.allclose(jac2.squeeze(-2), jac1)

print(torch.linalg.norm(jac2 - jac3))
assert torch.allclose(jac2, jac3)

print(torch.linalg.norm(jac2 - jac4))
assert torch.allclose(jac2, jac4)

print(torch.linalg.norm(jac2 - jac5))
assert torch.allclose(jac2, jac5)

print(torch.linalg.norm(jac4 - jac5))
assert torch.allclose(jac4, jac5)


torch.Size([100, 2])
torch.Size([100, 1, 2])
torch.Size([100, 1, 2])
torch.Size([100, 1, 2])
torch.Size([100, 1, 2])
tensor(0., device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)
tensor(0., device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)
tensor(8.1516e-08, device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)
tensor(7.7346e-08, device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)
tensor(6.6013e-08, device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)


In [16]:
t_forward = benchmark.Timer(
    stmt="mlp(x)",
    globals={"x": x, "mlp": mlp})

t1 = benchmark.Timer(
    stmt="func1(x, mlp)",
    setup="from __main__ import func1",
    globals={"x": x, "mlp": mlp})

t2 = benchmark.Timer(
    stmt="func2(x, mlp)",
    setup="from __main__ import func2",
    globals={"x": x, "mlp": mlp})

t3 = benchmark.Timer(
    stmt="func3(x, mlp)",
    setup="from __main__ import func3",
    globals={"x": x, "mlp": mlp})

t4 = benchmark.Timer(
    stmt="func4(x, mlp)",
    setup="from __main__ import func4",
    globals={"x": x, "mlp": mlp})

t5 = benchmark.Timer(
    stmt="func5(x, mlp)",
    setup="from __main__ import func5",
    globals={"x": x, "mlp": mlp})


print(t_forward.timeit(1000))
print(t1.timeit(1000))
print(t2.timeit(1000))
print(t3.timeit(1000))
print(t4.timeit(1000))
print(t5.timeit(1000))


mlp(x)
  100.94 us
  1 measurement, 1000 runs , 1 thread
func1(x, mlp)
setup: from __main__ import func1
  904.55 us
  1 measurement, 1000 runs , 1 thread
func2(x, mlp)
setup: from __main__ import func2
  1.17 ms
  1 measurement, 1000 runs , 1 thread
func3(x, mlp)
setup: from __main__ import func3
  853.16 us
  1 measurement, 1000 runs , 1 thread
func4(x, mlp)
setup: from __main__ import func4
  1.79 ms
  1 measurement, 1000 runs , 1 thread
func5(x, mlp)
setup: from __main__ import func5
  1.47 ms
  1 measurement, 1000 runs , 1 thread


## Compute Jacobian manually

In [ ]:
def get_activation_derivative(activation):
    if isinstance(activation, nn.Tanh):
        return lambda x: 1 - torch.tanh(x)**2
    elif isinstance(activation, nn.ELU):
        return lambda x: (x<=0) * activation.alpha * torch.exp(x) + (x>0) * 1.
    elif isinstance(activation, nn.ReLU):
        return lambda x: (x<=0) * 0. + (x>0) * 1.
    elif isinstance(activation, nn.LeakyReLU):
        return lambda x: (x<=0) * activation.negative_slope + (x>0) * 1.
    elif isinstance(activation, nn.Sigmoid):
        return lambda x: torch.sigmoid(x) - torch.sigmoid(x)**2
    else:
        return None


def calc_jacobian_1hidden(inputs, net, act_deriv=None, forward=False):
    """Calculate jacobian of a feedforward network with one hidden layer w.r.t. inputs.
    
    y = W2 * a
    a = activation(z)
    z = W1 * x + b1 
    
    -> dydx = dyda * dadz * dzdx 
            = W2 * dadz * W1
    """
    # inputs shape: (bs, n_in)
    # net(inputs) shape: (bs, n_out)
    # net: MLP with 1 hidden layer

    layer1, layer2 = net.layers
    act = net.activation
    z = layer1(inputs)  # (bs, n_hidden)
    
    if act_deriv is not None:
        dadz = act_deriv(z)  # (bs, n_hidden)
    else:  # compute using autodiff, about 10 times slower than explicit computation
        dadz = torch.diagonal(torch.func.vmap(torch.func.jacrev(act))(z), dim1=-2, dim2=-1)  # (bs, n_hidden)

    if forward:  # forward mode 
        dadx = dadz[:, :, None] * layer1.weight   # (bs, n_hidden, 1) * (n_hidden, n_in) -> (bs, n_hidden, n_in)
        dydx = torch.matmul(layer2.weight, dadx)  # (n_out, n_hidden) @ (bs, n_hidden, n_in) -> (bs, n_out, n_in)
    else:  # reverse mode
        dydz = dadz[:, None, :] * layer2.weight  # (bs, 1, n_hidden) * (n_out, n_hidden) -> (bs, n_out, n_hidden)
        dydx = torch.matmul(dydz, layer1.weight)  # (bs, n_out, n_hidden) @ (n_hidden, n_in) -> (bs, n_out, n_in)
        
    return dydx  # (bs, n_out, n_in)


def calc_jacobian_2hidden(inputs, net, act_deriv=None, forward=False):
    """Calculate jacobian of a feedforward network with two hidden layers w.r.t. inputs.
    
    y = W3 * a2
    a2 = activation(z2)
    z2 = W2 * a1 + b2
    a1 = activation(z1)
    z1 = W1 * x + b1
    
    -> dydx = W3 * da2dz2 * W2 * da1dz1 * W1
    """
    # inputs shape: (bs, n_in)
    # net(inputs) shape: (bs, n_out)
    # net: MLP with 2 hidden layers

    layer1, layer2, layer3 = net.layers
    act = net.activation

    z1 = layer1(inputs)  # (bs, n_hidden1)
    a1 = act(z1)
    z2 = layer2(a1)  # (bs, n_hidden2)

    if act_deriv is not None:
        da1dz1 = act_deriv(z1)  # (bs, n_hidden1)
        da2dz2 = act_deriv(z2)  # (bs, n_hidden2)
    else:
        da1dz1 = torch.diagonal(torch.func.vmap(torch.func.jacrev(act))(z1), dim1=-2, dim2=-1)  # (bs, n_hidden1)
        da2dz2 = torch.diagonal(torch.func.vmap(torch.func.jacrev(act))(z2), dim1=-2, dim2=-1)  # (bs, n_hidden2)

    if forward:
        dz1dx = layer1.weight
        da1dx = da1dz1[:, :, None] * dz1dx
        dz2dx = torch.matmul(layer2.weight, da1dx)
        da2dx = da2dz2[:, :, None] * dz2dx
        dydx = torch.matmul(layer3.weight, da2dx)
    else:
        dyda2 = layer3.weight  # (n_out, n_hidden2)
        dydz2 = da2dz2[:, None, :] * dyda2  # (bs, 1, n_hidden2) * (n_out, n_hidden2) -> (bs, n_out, n_hidden2)
        dyda1 = torch.matmul(dydz2, layer2.weight)  # (bs, n_out, n_hidden2) @ (n_hidden2, n_hidden1) -> (bs, n_out, n_hidden1)
        dydz1 = da1dz1[:, None, :] * dyda1  # (bs, 1, n_hidden1) * (bs, n_out, n_hidden1) -> (bs, n_out, n_hidden1)
        dydx = torch.matmul(dydz1, layer1.weight)  # (bs, n_out, n_hidden1) @ (n_hidden1, n_in) -> (bs, n_out, n_in)

    return dydx  # (bs, n_out, n_in)


def calc_jacobian_forward(inputs, net, act_deriv=None):
    """Calculate Jacobian of a multi-layer perceptron outputs w.r.t. inputs using forward mode.
    
    Example: 
    y = W3 * a2
    a2 = activation(z2)
    z2 = W2 * a1 + b2
    a1 = activation(z1)
    z1 = W1 * x + b1
    
    -> dydx = dydz1 * dz1dx 
            = dyda1 * da1dz1 * dz1dx 
            = ...
            = W3 * da2dz2 * W2 * da1dz1 * W1
    """

    act = net.activation
    n_layers = len(net.layers)

    current_jac = None

    for i in range(n_layers-1):
        
        # Update Jacobian for weight matrix of the current layer
        layer = net.layers[i]
        z = layer(inputs)
        if current_jac is None:  
            # first layer
            # shape: (n_hidden, n_in)
            current_jac = layer.weight  
        else:  
            # subsequent layers
            # shape: (bs, n_hidden, n_in) = (n_hidden, n_hidden) @ (bs, n_hidden, n_in)
            current_jac = torch.matmul(layer.weight, current_jac)  
            
        # Update Jacobian for activation
        inputs = act(z)
        if act_deriv is not None:
            dadz = act_deriv(z)  # shape: (bs, n_hidden)
        else:
            dadz = torch.diagonal(torch.func.vmap(torch.func.jacrev(act))(z), dim1=-2, dim2=-1)
        # shape: (bs, n_hidden, n_in) = (bs, n_hidden, 1) * (bs, n_hidden, n_in)
        current_jac = dadz[:, :, None] * current_jac
        
    # Update Jacobian for weight matrix of the last layer
    # shape: (bs, n_out, n_in) = (n_out, n_hidden) @ (bs, n_hidden, n_in)
    current_jac = torch.matmul(net.layers[-1].weight, current_jac)
    
    return current_jac  # (bs, n_out, n_in)


def calc_jacobian_reverse(inputs, net, act_deriv=None):
    """Calculate Jacobian of a multi-layer perceptron outputs w.r.t. inputs using reverse mode.
    
    Example:
    y = W3 * a2
    a2 = activation(z2)
    z2 = W2 * a1 + b2
    a1 = activation(z1)
    z1 = W1 * x + b1
    
    -> dydx = dyda2 * da2dx 
            = dyda2 * da2dz2 * dz2dx 
            = ...
            = W3 * da2dz2 * W2 * da1dz1 * W1
    """

    act = net.activation
    n_layers = len(net.layers)

    # Compute and save intermediate states 
    zs = []
    for i in range(n_layers-1):
        z = net.layers[i](inputs)
        inputs = act(z)
        zs.append(z)

    # Initialize Jacobian
    # shape: (n_out, n_hidden)
    current_jac = net.layers[-1].weight
        
    for i in range(n_layers-1):
        layer, z = net.layers[-2-i], zs[-1-i]
        if act_deriv is not None:
            dadz = act_deriv(z)  # shape: (bs, n_hidden)
        else:
            dadz = torch.diagonal(torch.func.vmap(torch.func.jacrev(act))(z), dim1=-2, dim2=-1)
        
        # Update Jacobian for activation
        # shape: (bs, n_out, n_hidden) = (bs, 1, n_hidden) * (bs, n_out, n_hidden)
        current_jac = dadz[:, None, :] * current_jac
        
        # Update Jacobian for weight matrix of the current layer
        # shape: for hidden layers (bs, n_out, n_hidden) = (bs, n_out, n_hidden) @ (n_hidden, n_hidden)
        #        for input layer   (bs, n_out, n_in) = (bs, n_out, n_hidden) @ (n_hidden, n_in)
        current_jac = torch.matmul(current_jac, layer.weight)

    return current_jac  # (bs, n_out, n_in)

In [33]:
def calc_jacobian(inputs, net, act_deriv):
    if len(net.layers) == 2:
        return calc_jacobian_1hidden(inputs, net, act_deriv, forward=False)
    elif len(net.layers) == 3:
        return calc_jacobian_2hidden(inputs, net, act_deriv, forward=False)
    else:
        return calc_jacobian_reverse(inputs, net, act_deriv)

In [58]:
x = torch.randn(100, 2, device=device, requires_grad=False)

mlp = MLP(layer_sizes=[2, 100, 1], activation=nn.Tanh())
mlp.to(device)
act_deriv = get_activation_derivative(mlp.activation)
mlp

MLP(
  (layers): ModuleList(
    (0): Linear(in_features=2, out_features=100, bias=True)
    (1): Linear(in_features=100, out_features=1, bias=True)
  )
  (activation): Tanh()
)

In [59]:
jac_fwd = calc_jacobian_forward(x, mlp, act_deriv=act_deriv)
jac_rev = calc_jacobian_reverse(x, mlp, act_deriv=act_deriv)
jac_fwd_2 = calc_jacobian_1hidden(x, mlp, act_deriv=act_deriv, forward=True)
jac_rev_2 = calc_jacobian_1hidden(x, mlp, act_deriv=act_deriv, forward=False)
jac = func3(x, mlp)

print(jac_fwd.shape)
print(jac_rev.shape)
print(jac.shape)

print(torch.linalg.norm(jac_fwd - jac))
assert torch.allclose(jac_fwd, jac, atol=1e-5)

print(torch.linalg.norm(jac_rev - jac))
assert torch.allclose(jac_rev, jac, atol=1e-5)

print(torch.linalg.norm(jac_fwd_2 - jac))
assert torch.allclose(jac_fwd_2, jac, atol=1e-5)

print(torch.linalg.norm(jac_rev_2 - jac))
assert torch.allclose(jac_rev_2, jac, atol=1e-5)

torch.Size([100, 1, 2])
torch.Size([100, 1, 2])
torch.Size([100, 1, 2])
tensor(4.9211e-07, device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)
tensor(4.7097e-07, device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)
tensor(4.9211e-07, device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)
tensor(4.7097e-07, device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)


In [61]:
t_forward = benchmark.Timer(
    stmt="mlp(x)",
    globals={"x": x, "mlp": mlp})

t_jac = benchmark.Timer(
    stmt="func3(x, mlp)",
    setup="from __main__ import func3",
    globals={"x": x, "mlp": mlp})

t_jac_fwd = benchmark.Timer(
    stmt="calc_jacobian_forward(x, mlp, act_deriv=act_deriv)",
    setup="from __main__ import calc_jacobian_forward",
    globals={"x": x, "mlp": mlp, "act_deriv": act_deriv})

t_jac_rev = benchmark.Timer(
    stmt="calc_jacobian_reverse(x, mlp, act_deriv=act_deriv)",
    setup="from __main__ import calc_jacobian_reverse",
    globals={"x": x, "mlp": mlp, "act_deriv": act_deriv})

t_jac_fwd_2 = benchmark.Timer(
    stmt="calc_jacobian_1hidden(x, mlp, act_deriv=act_deriv, forward=True)",
    setup="from __main__ import calc_jacobian_1hidden",
    globals={"x": x, "mlp": mlp, "act_deriv": act_deriv})

t_jac_rev_2 = benchmark.Timer(
    stmt="calc_jacobian_1hidden(x, mlp, act_deriv=act_deriv, forward=False)",
    setup="from __main__ import calc_jacobian_1hidden",
    globals={"x": x, "mlp": mlp, "act_deriv": act_deriv})



print(t_forward.timeit(1000))
print(t_jac.timeit(1000))
print(t_jac_fwd.timeit(1000))
print(t_jac_rev.timeit(1000))
print(t_jac_fwd_2.timeit(1000))
print(t_jac_rev_2.timeit(1000))


mlp(x)
  96.56 us
  1 measurement, 1000 runs , 1 thread
func3(x, mlp)
setup: from __main__ import func3
  882.23 us
  1 measurement, 1000 runs , 1 thread
calc_jacobian_forward(x, mlp, act_deriv=act_deriv)
setup: from __main__ import calc_jacobian_forward
  214.31 us
  1 measurement, 1000 runs , 1 thread
calc_jacobian_reverse(x, mlp, act_deriv=act_deriv)
setup: from __main__ import calc_jacobian_reverse
  160.59 us
  1 measurement, 1000 runs , 1 thread
calc_jacobian_1hidden(x, mlp, act_deriv=act_deriv, forward=True)
setup: from __main__ import calc_jacobian_1hidden
  163.34 us
  1 measurement, 1000 runs , 1 thread
calc_jacobian_1hidden(x, mlp, act_deriv=act_deriv, forward=False)
setup: from __main__ import calc_jacobian_1hidden
  131.72 us
  1 measurement, 1000 runs , 1 thread


In [55]:
jac = calc_jacobian(x, mlp, act_deriv=act_deriv)
loss = torch.nn.functional.mse_loss(jac.squeeze(-2), x)
loss.backward()

for l in mlp.layers:
    print(l.weight.grad)
    print(l.bias.grad)


tensor([[ 2.4245e-03, -2.4814e-03],
        [ 8.5505e-03, -2.0041e-02],
        [-2.9601e-04, -2.8361e-03],
        [-3.6414e-03,  8.9785e-03],
        [ 1.7384e-03,  7.8252e-03],
        [ 1.1040e-03, -3.3326e-03],
        [ 3.8284e-03,  6.8225e-03],
        [ 1.8925e-03, -3.1317e-03],
        [-1.2780e-04,  5.7422e-03],
        [ 1.7329e-03, -9.6979e-04],
        [-5.2253e-03, -8.3109e-04],
        [ 9.7290e-03,  3.9136e-03],
        [ 1.0932e-03, -1.0459e-03],
        [ 2.0616e-03, -7.1457e-04],
        [ 8.2028e-04, -1.5604e-03],
        [ 4.5132e-03, -8.9281e-03],
        [ 5.6910e-03,  1.0045e-04],
        [ 6.9668e-03, -2.8201e-03],
        [ 6.7537e-04, -2.2327e-02],
        [-1.2866e-03,  2.6214e-03],
        [ 8.6136e-04, -3.1281e-03],
        [-6.5844e-04,  7.2104e-03],
        [-8.0575e-04,  1.7875e-02],
        [-2.8448e-03, -2.4327e-03],
        [-3.7370e-04, -1.3262e-02],
        [ 2.5651e-03, -3.4748e-03],
        [-3.1100e-03,  1.1293e-03],
        [-6.5453e-04, -3.519

In [57]:
p = torch.randn(100, 2, device=device, requires_grad=False)
q = torch.randn(100, 2, device=device, requires_grad=False)

shift = nn.Parameter(torch.zeros(2), requires_grad=True).to(device)

def f(p, q):
    grad_V = calc_jacobian(p, mlp, act_deriv=act_deriv)
    grad_V = grad_V.squeeze(dim=-1)
    p_new = -q + grad_V
    q_new = p + shift 
    return 

t0 = benchmark.Timer(
    stmt="""
    f(p, q)
    """,
    setup="from __main__ import f",
    globals={"p": p, "q": q})

t0.timeit(100)

f(p, q)
setup: from __main__ import f
  398.87 us
  1 measurement, 100 runs , 1 thread

## Compute partial derivative

In [43]:
def func1(x, h, net):
    return vmap(jacrev(net, argnums=1))(x, h)  # (bs, n_out, n_in)

def func2(x, h, net):
    jac = jacrev(net, argnums=1)(x, h)  # (bs, n_out, bs, n_in)
    jac = torch.diagonal(jac, dim1=0, dim2=2)  # (n_out, n_in, bs)
    jac = torch.permute(jac, (2, 0, 1))  # (bs, n_out, n_in)
    return jac

def func3(x, h, net):
    return vmap(jacfwd(net, argnums=1))(x, h)  # (bs, n_out, n_in)

def func4(x, h, net):
    jac = jacfwd(net, argnums=1)(x, h)  # (bs, n_out, bs, n_in)
    jac = torch.diagonal(jac, dim1=0, dim2=2)  # (n_out, n_in, bs)
    jac = torch.permute(jac, (2, 0, 1))  # (bs, n_out, n_in)
    return jac

In [44]:
x = torch.randn(100, 2, device=device, requires_grad=False)
h = torch.randn(100, 1, device=device, requires_grad=False)

def net(x, h):
    return x + h * x

In [45]:
jac1 = func1(x, h, net)
jac2 = func2(x, h, net)
jac3 = func3(x, h, net)
jac4 = func4(x, h, net)

print(jac1.shape)
print(jac2.shape)
print(jac3.shape)
print(jac4.shape)

print(torch.linalg.norm(jac2.squeeze(-2) - jac1))
assert torch.allclose(jac2.squeeze(-2), jac1)

print(torch.linalg.norm(jac2 - jac3))
assert torch.allclose(jac2, jac3)

print(torch.linalg.norm(jac2 - jac4))
assert torch.allclose(jac2, jac4)

torch.Size([100, 2, 1])
torch.Size([100, 2, 1])
torch.Size([100, 2, 1])
torch.Size([100, 2, 1])
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')


In [52]:
assert torch.allclose(jac1.squeeze(-1), x)

In [46]:
t_forward = benchmark.Timer(
    stmt="net(x, h)",
    globals={"x": x, "h": h, "net": net})

t1 = benchmark.Timer(
    stmt="func1(x, h, net)",
    setup="from __main__ import func1",
    globals={"x": x, "h": h, "net": net})

t2 = benchmark.Timer(
    stmt="func2(x, h, net)",
    setup="from __main__ import func2",
    globals={"x": x, "h": h, "net": net})

t3 = benchmark.Timer(
    stmt="func3(x, h, net)",
    setup="from __main__ import func3",
    globals={"x": x, "h": h, "net": net})

t4 = benchmark.Timer(
    stmt="func4(x, h, net)",
    setup="from __main__ import func4",
    globals={"x": x, "h": h, "net": net})


print(t_forward.timeit(1000))
print(t1.timeit(1000))
print(t2.timeit(1000))
print(t3.timeit(1000))
print(t4.timeit(1000))


net(x, h)
  26.47 us
  1 measurement, 1000 runs , 1 thread
func1(x, h, net)
setup: from __main__ import func1
  528.87 us
  1 measurement, 1000 runs , 1 thread
func2(x, h, net)
setup: from __main__ import func2
  488.92 us
  1 measurement, 1000 runs , 1 thread
func3(x, h, net)
setup: from __main__ import func3
  1.03 ms
  1 measurement, 1000 runs , 1 thread
func4(x, h, net)
setup: from __main__ import func4
  932.44 us
  1 measurement, 1000 runs , 1 thread
